In [25]:
import os
from dotenv import load_dotenv

In [26]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_community.embeddings import FakeEmbeddings


In [27]:
load_dotenv()


True

In [28]:
def load_and_chunk_pdf(pdf_path):
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    return splitter.split_documents(documents)

In [29]:
def create_vectorstore(chunks):
    embeddings = FakeEmbeddings(size=384)
    return FAISS.from_documents(chunks, embeddings)

In [30]:
def retrieve_context(vectorstore, question, k=3):
    docs = vectorstore.similarity_search(question, k=k)
    context = "\n\n".join([doc.page_content for doc in docs])
    return context, docs

In [31]:
def ask_llm(context, question):
    llm = ChatGroq(
        model_name="openai/gpt-oss-120b",
        temperature=0
    )

    prompt = f"""
Answer the question using ONLY the context below.
If the answer is not present, say "Not found in document".

Context:
{context}

Question:
{question}
"""

    response = llm.invoke(prompt)
    return response.content

In [32]:
if __name__ == "__main__":
    print("\n📄 Chat with Your PDF (RAG – No Chaining)\n")

    pdf_path = "check.pdf"

    print("⏳ Indexing document...")
    chunks = load_and_chunk_pdf(pdf_path)
    print(chunks)
    vectorstore = create_vectorstore(chunks)


    question = "How to install the GVC client?"
    context, docs = retrieve_context(vectorstore, question)
    answer = ask_llm(context, question)
    print(answer)


📄 Chat with Your PDF (RAG – No Chaining)

⏳ Indexing document...
[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-03-19T12:11:25+05:30', 'author': 'Krishnadass M.', 'moddate': '2024-03-19T12:11:25+05:30', 'source': 'check.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1'}, page_content='How to Setup Global VPN Client (GVC) \nIn this document, we shall see the following: \n \n1. Installing the VPN client software on your laptop/PC. \n2. Connecting to the campus network using the VPN client software. \n \nImportant Note to Windows 7 users : Please ensure that you have installed the Windows 7 update patch \nbefore installing GVC Client.  Download the appropriate patch from the following location: \n64 Bit Patch :https://intranet.cb.amrita.edu/download/VPN/Windows6.1-KB3033929-x64.msu \n32Bit Patch : https://intranet.cb.amrita.edu/download/VPN/Windows6.1-KB3033929-x86.msu \n \nINSTALLING THE GVC CLIENT \nTo install GVC usi